In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="POSIST",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="discount",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
from pyspark.sql.functions import *
import re
import time

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

# Usage:
# bill_snake = to_snake_case_df(bill)
# discount_bil = enrich_json(bill_snake)

def enrich_json(df_json):

    df_map = (
        spark.read.table('fq_dev_catalog.silver.dim_store')
        .select(
            col('currency_code'),
            col('store_id'),
            col('store_name'),
             col('brand_id'),
            col('company_id'),
            # col('store_location')
        ).distinct()
    )

    df_enriched = (
        df_json.join(
            df_map,
            df_json['deployment_name'] == df_map['store_name'],
            'left'
        )
        # .drop(df_map.deployment_name)
        .drop(df_json.deployment_name)
    )
    return df_enriched

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_catalog.silver.discount_bill(
    bill_total DOUBLE,
    business_date DATE,
    created_time STRING,
    date_of_transaction DATE,
    discount_name STRING,
    discount_value DOUBLE,
    tab_name STRING,
    source_system STRING,
    sys_id STRING,
    deployment_name STRING
)
PARTITIONED BY ( -- low cardinality column first
    deployment_name,
    business_date
)
TBLPROPERTIES (
    delta.columnMapping.mode = 'name',
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
);

In [0]:
def merge_stream_bill(df, i):
    try:
        exploded_df = (
            df.selectExpr(
                # "file_name",
                "explode(Discount) as discount"
            ).select(
                # "file_name",
                'discount.*'
            ).withColumn(
                "source_system", lit(f"{source}")
            ).withColumn(
                'sys_id', expr("uuid()")
            )
        )
        discount_bill = to_snake_case_df(exploded_df)
        # discount_bill = enrich_json(exploded_df)
        discount_bill.createOrReplaceTempView("discount_bill_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.discount_bill target
            USING (
                SELECT
                    *
                FROM discount_bill_upsert_microbatch
            ) as source
            ON target.business_date = source.business_date
               AND target.deployment_name = source.deployment_name
            --    AND target.file_name = source.file_name
            WHEN MATCHED THEN DELETE
        """)

        df.sparkSession.sql("""
                MERGE INTO fq_dev_catalog.silver.discount_bill target
                USING (
                SELECT
                    *
                FROM discount_bill_upsert_microbatch
                ) as source
                ON target.business_date = source.business_date
                AND target.deployment_name = source.deployment_name
                --    AND target.file_name = source.file_name
                WHEN NOT MATCHED THEN INSERT *
        """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    .table("fq_dev_catalog.bronze.discount")
    .writeStream
    .foreachBatch(merge_stream_bill)
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_discount_bill')  
    .trigger(availableNow=True)
    .start()
)

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(DISTINCT business_date) AS business_date_cardinality,
  COUNT(DISTINCT deployment_name) AS deployment_name_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.discount_bill;

In [0]:
%sql
-- OPTIMIZE fq_dev_catalog.silver.discount_bill --ZORDER BY (); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.silver.dim_tab

In [0]:
%sql
select * from fq_dev_catalog.silver.discount_bill

In [0]:
%sql
select count(*) from fq_dev_catalog.silver.discount_bill where business_date = '2026-01-19';

In [0]:
%sql
select count(*) from fq_dev_catalog.bronze.discount LATERAL VIEW explode(Discount) AS d WHERE d.`Business Date` = '2026-01-19';

In [0]:
for query in spark.streams.active:
    query.stop()